# PyDI Data Integration Workflow: Music

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with music datasets to showcase the data integration pipeline from schema and entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Schema Matching and Value Normalization](#part-1-schema-matching-and-value-normalization)
- [Part 2: Data Loading and Profiling](#part-2-data-loading-and-profiling)
- [Part 3: Entity Matching](#part-3-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 4: Data Fusion](#part-4-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

## Part 1: Schema Matching and Value Normalization

In [1]:
from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd
import json
from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.normalization import load_normalization_spec
from langchain_openai import ChatOpenAI

/Users/aaronsteiner/Documents/GitHub/PyDI/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Target Schema and Normalization Spec

In [3]:
# Load the JSON Schema (used for both matching and normalization)
with open(INPUT_DIR / "schemamatching" / "target_schema.json") as f:
    target_schema = json.load(f)

# Load NormalizationSpec from the same schema
spec = load_normalization_spec(INPUT_DIR / "schemamatching" / "target_schema.json")

spec.set_column("tracks", output_type="list")

target_columns = list(spec.columns.keys())

# Create empty target DataFrame for schema matching
df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Show column types derived from schema
pd.DataFrame([
    {"column": col, "output_type": col_spec.output_type}
    for col, col_spec in spec.columns.items()
])

,column,output_type
0,id,string
1,name,string
2,artist,string
3,release-date,datetime
4,release-country,string
5,label,string
6,genre,string
7,duration,int
8,tracks,list


## Step 2: Load Source Datasets

In [4]:
from PyDI.io import load_xml, load_csv, load_json
mbrainz = load_csv(INPUT_DIR / "data" / "musicbrainz.csv")
mbrainz.attrs["dataset_name"] = "musicbrainz"
mbrainz.head()

,id,name,artist,release-date,release-country,duration,label,genre,tracks
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom of Great Britain and Northern I...,1055,NaN,NaN,"['Fermats Theorem', 'Sight Beyond']"
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom of Great Britain and Northern I...,724,NaN,NaN,"['Tempest', 'Inner Sense']"
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States of America,2384,NaN,NaN,"[""The Sign's Alive (original mix)"", ""The Sign'..."
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States of America,1626,NaN,NaN,"['Surrender (Petalpusher original)', ""Surrende..."
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,4145,NaN,NaN,"['The Warning', 'City Sphere', 'Forgotten Thou..."


In [5]:
import re
# Clean column names from XML namespaces
def strip_all_ns(col):
    """
    Remove ALL {namespace} prefixes from a column name string.
    Example:
      '{ns}medium-list_{ns}medium_{ns}track' -> 'medium-list_medium_track'
    """
    if not isinstance(col, str):
        return col
    # remove every occurrence of {...}
    return re.sub(r"\{[^}]+\}", "", col)

mbrainz = mbrainz.rename(columns=strip_all_ns)
mbrainz.head()

,id,name,artist,release-date,release-country,duration,label,genre,tracks
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom of Great Britain and Northern I...,1055,NaN,NaN,"['Fermats Theorem', 'Sight Beyond']"
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom of Great Britain and Northern I...,724,NaN,NaN,"['Tempest', 'Inner Sense']"
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States of America,2384,NaN,NaN,"[""The Sign's Alive (original mix)"", ""The Sign'..."
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States of America,1626,NaN,NaN,"['Surrender (Petalpusher original)', ""Surrende..."
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,4145,NaN,NaN,"['The Warning', 'City Sphere', 'Forgotten Thou..."


In [6]:
lastfm = load_csv(INPUT_DIR / "data" / "lastfm.csv")
lastfm.attrs["dataset_name"] = "lastfm"
lastfm.head()

,id,name,artist,release-date,release-country,duration,label,genre,tracks
0,lastFM_1,John B - Fermats Theorem / Sight Beyond,John B,NaN,NaN,903.0,NaN,NaN,"['Fermats Theorem', 'Sight Beyond']"
1,lastFM_2,Tempest / Inner Sense,Psychosis,NaN,NaN,734.0,NaN,NaN,"['Tempest', 'Inner Sense']"
2,lastFM_4,Petalpusher - Surrender,Petalpusher,NaN,NaN,1626.0,NaN,NaN,"['Surrender (Petalpusher Original)', ""Surrende..."
3,lastFM_8,in the spirit,R. Trent,NaN,NaN,1265.0,NaN,NaN,"['In The Spirit (The Full Experience)', 'In Th..."
4,lastFM_11,Come Of Age,S. Vitus Dance,NaN,NaN,1378.0,NaN,NaN,"['Bliss', 'Tunnel Vision', 'Catch The Sun', 'M..."


In [7]:
discogs = load_csv(INPUT_DIR / "data" / "discogs.csv")
discogs.attrs["dataset_name"] = "discogs"
discogs.head()

,id,name,artist,release-date,release-country,duration,label,genre,tracks
0,discogs_3,Fermats Theorem / Sight Beyond,John B,1996-01-01,UK,0,New Identity Recordings,Electronic,"['Fermats Theorem', 'Sight Beyond']"
1,discogs_4,Tempest / Inner Sense,Psychosis,1998-01-01,UK,0,Renegade Hardware,Electronic,"['Tempest', 'Inner Sense']"
2,discogs_5,The Sign's Alive,Lypid,2000-09-05,United States of America,0,Statra Recordings,Electronic,"[""The Sign's Alive (Original Mix)"", ""The Sign'..."
3,discogs_6,Surrender,Petalpusher,1999-04-27,United States of America,1626,Naked Music Recordings,Electronic,"['Surrender (Petalpusher Original)', ""Surrende..."
4,discogs_11,Unreasonable Behaviour,Laurent Garnier,2000-06-01,France,5938,F Communications,Electronic,"['The Warning', 'City Sphere', 'Forgotten Thou..."


## Step 3: LLM-Based Schema Matching

In [8]:
from dotenv import load_dotenv
load_dotenv()

# Initialize matcher with target schema for better context
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5"),
    num_rows=40,
    target_schema=target_schema,
)

# Match mbrainz dataset
mbrainz_mapping = matcher.match(mbrainz, df_target)

mbrainz_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,musicbrainz,id,target_schema,id,0.95,llm_based_matching
1,musicbrainz,name,target_schema,name,0.95,llm_based_matching
2,musicbrainz,artist,target_schema,artist,0.95,llm_based_matching
3,musicbrainz,release-date,target_schema,release-date,0.95,llm_based_matching
4,musicbrainz,release-country,target_schema,release-country,0.95,llm_based_matching
5,musicbrainz,duration,target_schema,duration,0.95,llm_based_matching
6,musicbrainz,label,target_schema,label,0.95,llm_based_matching
7,musicbrainz,genre,target_schema,genre,0.95,llm_based_matching
8,musicbrainz,tracks,target_schema,tracks,0.95,llm_based_matching


In [9]:
lastfm_mapping = matcher.match(lastfm, df_target)
lastfm_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,lastfm,id,target_schema,id,0.95,llm_based_matching
1,lastfm,name,target_schema,name,0.95,llm_based_matching
2,lastfm,artist,target_schema,artist,0.95,llm_based_matching
3,lastfm,release-date,target_schema,release-date,0.95,llm_based_matching
4,lastfm,release-country,target_schema,release-country,0.95,llm_based_matching
5,lastfm,duration,target_schema,duration,0.95,llm_based_matching
6,lastfm,label,target_schema,label,0.95,llm_based_matching
7,lastfm,genre,target_schema,genre,0.95,llm_based_matching
8,lastfm,tracks,target_schema,tracks,0.95,llm_based_matching


In [10]:
discogs_mapping = matcher.match(discogs, df_target)
discogs_mapping

,source_dataset,source_column,target_dataset,target_column,score,notes
0,discogs,id,target_schema,id,0.95,llm_based_matching
1,discogs,name,target_schema,name,0.95,llm_based_matching
2,discogs,artist,target_schema,artist,0.95,llm_based_matching
3,discogs,release-date,target_schema,release-date,0.95,llm_based_matching
4,discogs,release-country,target_schema,release-country,0.95,llm_based_matching
5,discogs,duration,target_schema,duration,0.95,llm_based_matching
6,discogs,label,target_schema,label,0.95,llm_based_matching
7,discogs,genre,target_schema,genre,0.95,llm_based_matching
8,discogs,tracks,target_schema,tracks,0.95,llm_based_matching


## Step 4: Translate and Normalize


In [11]:
# Handle 00 days and months in discogs
discogs["release-date"] = discogs["release-date"].apply(lambda x: re.sub(r"-00", "-01", x) if isinstance(x, str) else x)
discogs.iloc[0]["release-date"]

'1996-01-01'

In [12]:
translator = SchemaTranslator()

# Translate + normalize each dataset with its own mapping

spec.set_column("release-country", country_format="name")
spec.set_column("release-date", output_type="datetime")

discogs_normalized = translator.translate(
    discogs,
    discogs_mapping,
    normalize=spec,
    on_failure="keep"
)

lastfm_normalized = translator.translate(
    lastfm, lastfm_mapping,
    normalize=spec, on_failure="keep"
)

mbrainz_normalized = translator.translate(
    mbrainz, mbrainz_mapping,
    normalize=spec, on_failure="keep"
)

In [13]:
# Inspect normalized mbrainz dataset (target columns only)
mbrainz_cols = [c for c in target_columns if c in mbrainz_normalized.columns]
mbrainz_normalized[mbrainz_cols].head(10)

,id,name,artist,release-date,release-country,label,genre,duration,tracks
0,mbrainz_1,Fermats Theorem / Sight Beyond,John B,1996-01-01,United Kingdom,NaN,NaN,1055,"['Fermats Theorem', 'Sight Beyond']"
1,mbrainz_2,Tempest / Inner Sense,Psychosis,1998-12-14,United Kingdom,NaN,NaN,724,"['Tempest', 'Inner Sense']"
2,mbrainz_3,The Sign's Alive,Lypid,2000-09-05,United States,NaN,NaN,2384,"[""The Sign's Alive (original mix)"", ""The Sign'..."
3,mbrainz_4,Surrender,Petalpusher,1999-04-27,United States,NaN,NaN,1626,"['Surrender (Petalpusher original)', ""Surrende..."
4,mbrainz_6,Unreasonable Behaviour,"Garnier, Laurent",2000-07-24,France,NaN,NaN,4145,"['The Warning', 'City Sphere', 'Forgotten Thou..."
5,mbrainz_7,In the Spirit,"Trent, Ron",1999-01-01,United Kingdom,NaN,NaN,1291,"['In the Spirit (The Full Experience)', 'In th..."
6,mbrainz_8,The Dance,"Gallery Collective, The",1996-01-01,United States,NaN,NaN,1869,"['The Dance (The Full Gallery mix)', 'The Danc..."
7,mbrainz_9,Come of Age,St. Vitus Dance,1994-01-01,United Kingdom,NaN,NaN,1463,"['Bliss', 'Tunnel Vision', 'Catch the Sun', 'M..."
8,mbrainz_10,Contrax / All Mighty,Decorum,1999-01-01,United Kingdom,NaN,NaN,779,"['Contrax', 'All Mighty']"
9,mbrainz_11,Electronically Tested,Surgeon,1995-01-01,United Kingdom,NaN,NaN,1613,"['Barrier Method', 'Pork Machine', 'Language B..."


In [14]:
discogs_cols = [c for c in target_columns if c in discogs_normalized.columns]
discogs_normalized[discogs_cols].head(10)

,id,name,artist,release-date,release-country,label,genre,duration,tracks
0,discogs_3,Fermats Theorem / Sight Beyond,John B,1996-01-01,Uganda,New Identity Recordings,Electronic,0,"['Fermats Theorem', 'Sight Beyond']"
1,discogs_4,Tempest / Inner Sense,Psychosis,1998-01-01,Uganda,Renegade Hardware,Electronic,0,"['Tempest', 'Inner Sense']"
2,discogs_5,The Sign's Alive,Lypid,2000-09-05,United States,Statra Recordings,Electronic,0,"[""The Sign's Alive (Original Mix)"", ""The Sign'..."
3,discogs_6,Surrender,Petalpusher,1999-04-27,United States,Naked Music Recordings,Electronic,1626,"['Surrender (Petalpusher Original)', ""Surrende..."
4,discogs_11,Unreasonable Behaviour,Laurent Garnier,2000-06-01,France,F Communications,Electronic,5938,"['The Warning', 'City Sphere', 'Forgotten Thou..."
5,discogs_13,In The Spirit,Ron Trent,1999-01-01,Uganda,Peacefrog Records,Electronic,0,"['In The Spirit (The Full Experience)', 'In Th..."
6,discogs_14,The Dance,The Gallery Collective,1996-01-01,United States,Prescription,Electronic,0,"['The Dance (The Full Gallery Mix)', 'The Danc..."
7,discogs_16,Analogue,Mampi Swift,1997-01-01,Uganda,Suburban Base Records,Electronic,0,"['Analogue', 'Behold']"
8,discogs_17,Come Of Age,St. Vitus Dance,1994-01-01,Uganda,Peacefrog Records,Electronic,0,"['Bliss', 'Tunnel Vision', 'Catch The Sun', 'M..."
9,discogs_18,Ebony Angel - The Resurrection,Monica Elam,1999-01-01,United States,Clairaudience,Electronic,0,"['The Resurrection (Rainforest Rhapsody)', 'Th..."


In [15]:
lastfm_cols = [c for c in target_columns if c in lastfm_normalized.columns]
lastfm_normalized[lastfm_cols].head(10)

,id,name,artist,release-date,release-country,label,genre,duration,tracks
0,lastFM_1,John B - Fermats Theorem / Sight Beyond,John B,NaN,NaN,NaN,NaN,903.0,"['Fermats Theorem', 'Sight Beyond']"
1,lastFM_2,Tempest / Inner Sense,Psychosis,NaN,NaN,NaN,NaN,734.0,"['Tempest', 'Inner Sense']"
2,lastFM_4,Petalpusher - Surrender,Petalpusher,NaN,NaN,NaN,NaN,1626.0,"['Surrender (Petalpusher Original)', ""Surrende..."
3,lastFM_8,in the spirit,R. Trent,NaN,NaN,NaN,NaN,1265.0,"['In The Spirit (The Full Experience)', 'In Th..."
4,lastFM_11,Come Of Age,S. Vitus Dance,NaN,NaN,NaN,NaN,1378.0,"['Bliss', 'Tunnel Vision', 'Catch The Sun', 'M..."
5,lastFM_16,Devotional,D. Alvarado,NaN,NaN,NaN,NaN,944.0,"['Devotional', 'Sunstone']"
6,lastFM_24,P. Johnson - The Music In Me,P. Johnson,NaN,NaN,NaN,NaN,1168.0,"['The Music In Me', 'Slinky', ""Y'All Stole The..."
7,lastFM_27,On and,GH-106,NaN,NaN,NaN,NaN,120.0,"['XJ6', 'Fantasy']"
8,lastFM_28,- new - Subtle Frequencies,C. Jackson,NaN,NaN,NaN,NaN,613.0,"['Check our Beats', 'Teleport']"
9,lastFM_30,Second Area / Think Tank,Inigo Kennedy,NaN,NaN,NaN,NaN,827.0,"['Second Area', 'Think Tank']"


In [16]:
# Only keep target columns
mbrainz = mbrainz_normalized[mbrainz_cols].copy()
lastfm = lastfm_normalized[lastfm_cols].copy()
discogs = discogs_normalized[discogs_cols].copy()

import ast

def parse_track_list(value):
    if isinstance(value, list):
        items = value
    elif pd.isna(value):
        return []
    elif isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            items = parsed if isinstance(parsed, list) else [parsed]
        except (SyntaxError, ValueError):
            items = [part.strip() for part in text.split("|")]
    else:
        items = [value]

    cleaned = []
    seen = set()
    for item in items:
        if item is None or pd.isna(item):
            continue
        title = str(item).strip()
        if not title:
            continue
        key = re.sub(r"\s+", " ", title.casefold())
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(title)
    return cleaned

for dataset in (mbrainz, discogs, lastfm):
    if "tracks" in dataset.columns:
        dataset["tracks"] = dataset["tracks"].apply(parse_track_list)


## Part 2: Data Loading and Profiling

In [17]:
# Display basic information
datasets = [discogs, mbrainz, lastfm]
names = ["Discogs", "MusicBrainz", "Last.fm"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 37,255


In [18]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

discogs:
  Rows: 22,627
  Columns: 9
  Total nulls: 2,834
  Null percentage: 1.4%
  Null counts per column:
    release-date: 2,234 (9.9%)
    release-country: 600 (2.7%)

musicbrainz:
  Rows: 4,763
  Columns: 9
  Total nulls: 10,587
  Null percentage: 24.7%
  Null counts per column:
    release-date: 313 (6.6%)
    release-country: 748 (15.7%)
    label: 4,763 (100.0%)
    genre: 4,763 (100.0%)

lastfm:
  Rows: 9,865
  Columns: 9
  Total nulls: 44,690
  Null percentage: 50.3%
  Null counts per column:
    release-date: 9,865 (100.0%)
    release-country: 9,865 (100.0%)
    label: 9,865 (100.0%)
    genre: 9,865 (100.0%)
    duration: 5,230 (53.0%)



{'rows': 9865,
 'columns': 9,
 'nulls_total': 44690,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'artist': 0,
  'release-date': 9865,
  'release-country': 9865,
  'label': 9865,
  'genre': 9865,
  'duration': 5230,
  'tracks': 0},
 'dtypes': {'id': 'object',
  'name': 'object',
  'artist': 'object',
  'release-date': 'float64',
  'release-country': 'float64',
  'label': 'float64',
  'genre': 'float64',
  'duration': 'float64',
  'tracks': 'object'}}

### Attribute Coverage Analysis

In [19]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print(" Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

 Attribute coverage across datasets:


,attribute,discogs_count,discogs_pct,discogs_coverage,discogs_samples,musicbrainz_count,musicbrainz_pct,musicbrainz_coverage,musicbrainz_samples,lastfm_count,lastfm_pct,lastfm_coverage,lastfm_samples,avg_coverage,max_coverage,datasets_with_attribute
0,artist,22627/22627,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",4763/4763,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",9865/9865,100.0%,1.000000,"['John B', 'Psychosis', 'Petalpusher']",1.000000,1.000000,3
1,duration,22627/22627,100.0%,1.000000,"[0, 0, 0]",4763/4763,100.0%,1.000000,"[1055, 724, 2384]",4635/9865,47.0%,0.469843,"[903.0, 734.0, 1626.0]",0.823281,1.000000,3
2,genre,22627/22627,100.0%,1.000000,"['Electronic', 'Electronic', 'Electronic']",0/4763,0.0%,0.000000,[],0/9865,0.0%,0.000000,[],0.333333,1.000000,1
3,id,22627/22627,100.0%,1.000000,"['discogs_3', 'discogs_4', 'discogs_5']",4763/4763,100.0%,1.000000,"['mbrainz_1', 'mbrainz_2', 'mbrainz_3']",9865/9865,100.0%,1.000000,"['lastFM_1', 'lastFM_2', 'lastFM_4']",1.000000,1.000000,3
4,label,22627/22627,100.0%,1.000000,"['New Identity Recordings', 'Renegade Hardware...",0/4763,0.0%,0.000000,[],0/9865,0.0%,0.000000,[],0.333333,1.000000,1
5,name,22627/22627,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",4763/4763,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",9865/9865,100.0%,1.000000,"['John B - Fermats Theorem / Sight Beyond', '...",1.000000,1.000000,3
6,release-country,22027/22627,97.3%,0.973483,"['Uganda', 'Uganda', 'United States']",4015/4763,84.3%,0.842956,"['United Kingdom', 'United Kingdom', 'United S...",0/9865,0.0%,0.000000,[],0.605480,0.973483,2
7,release-date,20393/22627,90.1%,0.901268,"[Timestamp('1996-01-01 00:00:00'), Timestamp('...",4450/4763,93.4%,0.934285,"[Timestamp('1996-01-01 00:00:00'), Timestamp('...",0/9865,0.0%,0.000000,[],0.611851,0.934285,2
8,tracks,22627/22627,100.0%,1.000000,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",4763/4763,100.0%,1.000000,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",9865/9865,100.0%,1.000000,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",1.000000,1.000000,3



 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['artist', 'duration', 'id', 'name', 'release-country', 'release-date', 'tracks']


### Detailed Data Profiling

In [20]:
from pathlib import Path

# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(datasets, names):
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")


Profiling Discogs...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 317.56it/s]


Profile saved: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/dataset-profiles/discogs_profile.html
Profiling MusicBrainz...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 348.68it/s]


Profile saved: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/dataset-profiles/musicbrainz_profile.html
Profiling Last.fm...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 611.59it/s]

Profile saved: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/dataset-profiles/lastfm_profile.html

 Generated 3 detailed HTML reports
 Location: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • discogs_profile.html
  • musicbrainz_profile.html
  • lastfm_profile.html


## Part 3: Entity Matching

### Step 1: Blocking

In [21]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [22]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - Longest Token in Name
# Add name_longest_token directly to the original dataframes
def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

mbrainz['name_longest_token'] = mbrainz['name'].apply(get_longest_token)
discogs['name_longest_token'] = discogs['name'].apply(get_longest_token)
lastfm['name_longest_token'] = lastfm['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    mbrainz, discogs,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

standard_blocker_m2l = StandardBlocker(
    mbrainz, lastfm,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4685 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1443 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2850 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1368 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/aaronsteiner/Documents/GitHub/PyDI

### Step 2: Evaluate Blocking Against Ground Truth

In [23]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 10 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 13 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 25 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 42 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 48 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 55 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 60 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 63 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 79 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 98 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 110 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 131 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 137 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 141 true matches
[INFO ]

{'pair_completeness': 1.0,
 'pair_quality': 0.002145844289888658,
 'reduction_ratio': 0.9967007230357613,
 'total_candidates': 355571,
 'total_possible_pairs': 107772401,
 'true_positives_found': 763,
 'total_true_pairs': 763,
 'batches_processed': 356,
 'evaluation_timestamp': '2026-05-06T18:14:51.127827',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/blocking-evaluation/blocking_detailed_results.csv']}

In [24]:


# Now evaluate blocking for musicbrainz and lastfm combination

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2l,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 17 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 40 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 51 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 65 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 93 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 114 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 143 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 181 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 202 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 238 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 269 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 337 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 377 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 406 true matches
[I

{'pair_completeness': 0.9292682926829269,
 'pair_quality': 0.004941345835846157,
 'reduction_ratio': 0.9967180493240736,
 'total_candidates': 154209,
 'total_possible_pairs': 46986995,
 'true_positives_found': 762,
 'total_true_pairs': 820,
 'batches_processed': 155,
 'evaluation_timestamp': '2026-05-06T18:14:55.258826',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [25]:
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators = [
    # Release name — Jaccard
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release artist — Jaccard
    StringComparator(
        column='artist',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release duration — within 10% --> allow 10% deviation
    NumericComparator(
        column='duration',
        method='relative_difference',
        max_difference=0.10
    ),
    # # Track list — overlap
    StringComparator(
        column='tracks',
        similarity_function='jaccard',
        preprocess=normalize_text,
        list_strategy="set_overlap"
    ),
    # Release date — within 2 years
    DateComparator(
        column='release-date',
        max_days_difference=365 * 2
    ),
    # Release country — Jaccard
    StringComparator(
        column='release-country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [26]:
import numpy as np

# Convert lists in mbrainz["duration"] to single integer values (sum if list, else int)

def sum_duration(val):
    if isinstance(val, list):
        return int(np.nansum([int(x) for x in val if str(x).isdigit()]))
    try:
        return int(val)
    except Exception:
        return np.nan

mbrainz["duration"] = mbrainz["duration"].apply(sum_duration)

In [27]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=mbrainz,
    df_right=discogs, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=None, # equal weights for all 4 comparators
    threshold=0.5,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4763 x 22627 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4763 x 22627 elements after 0:00:0.065; 355571 blocked pairs (reduction ratio: 0.9967007230357613)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:48.815; found 3897 correspondences.


In [28]:
matcher = RuleBasedMatcher()

# Remove release-date and release-country comparator since those columns are not present in lastfm dataset
comparators.pop()
comparators.pop()

correspondences_m2l = matcher.match(
    df_left=mbrainz,
    df_right=lastfm, 
    candidates=standard_blocker_m2l,
    comparators=comparators,
    weights=None, # equal weights for all 4 comparators
    threshold=0.3,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4763 x 9865 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4763 x 9865 elements after 0:00:0.028; 154209 blocked pairs (reduction ratio: 0.9967180493240736)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:17.196; found 4578 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [29]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  328
[INFO ] root -   True Negatives:  652
[INFO ] root -   False Positives: 15
[INFO ] root -   False Negatives: 5
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.980
[INFO ] root -   Precision: 0.956
[INFO ] root -   Recall:    0.985
[INFO ] root -   F1-Score:  0.970


{'precision': 0.956268221574344,
 'recall': 0.984984984984985,
 'f1': 0.970414201183432,
 'accuracy': 0.98,
 'true_positives': 328,
 'false_positives': 15,
 'false_negatives': 5,
 'true_negatives': 652,
 'threshold_used': 0.0,
 'total_correspondences': 3897,
 'filtered_correspondences': 3897,
 'evaluation_timestamp': '2026-05-06T18:16:03.154170',
 'output_files': ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/debug_results_entity_matching/matching_detailed_results.csv']}

In [30]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2992 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2461	|	82.25%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	350	|	11.70%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	107	|	3.58%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	37	|	1.24%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	19	|	0.64%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	8	|	0.27%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	2	|	0.07%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	2	|	0.07%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	3	|	0.10%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	1	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		14	|	1	|	0.03%
[INFO ] PyDI.entitymatching.evaluation - 		15	|	1	|	0.03%
[INFO ] root - Cluster size distribution wr

Analyzing cluster size distribution in our entity matching results...

📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,2461,82.252674
1,3,350,11.697861
2,4,107,3.576203
3,5,37,1.236631
4,6,19,0.635027
5,7,8,0.267380
6,8,2,0.066845
7,9,2,0.066845
8,10,3,0.100267
9,11,1,0.033422


In [31]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 2992 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [32]:
from PyDI.entitymatching import MaximumBipartiteMatching

# use Maximum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
correspondences_m2d_post = clusterer.cluster(correspondences_m2d)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d_post,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d_post,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

[INFO ] root - Filtered correspondences: 3897 -> 3897 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 3897 -> 3054 
[INFO ] root - MaximumBipartiteMatching: 3897 -> 3054 correspondences
[INFO ] root - MaximumBipartiteMatching: 6859 -> 6108 entities
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 3054 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	3054	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/cluster_analysis/cluster_size_distribution.csv
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  281
[INFO ] root -   True Negatives:  661
[INFO ] root -   False Positives: 6
[INFO ] root -   False Negatives: 52
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.942
[INFO

In [33]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

print("Eval results before clustering:")
for metric, value in eval_results.items():
    print(f"  {metric}: {value}")

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2l,
)

clusterer = MaximumBipartiteMatching()
correspondences_m2l_post = clusterer.cluster(correspondences_m2l)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2l_post,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l_post,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

print("Eval results after clustering:")
for metric, value in eval_results.items():
    print(f"  {metric}: {value}")

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  305
[INFO ] root -   True Negatives:  654
[INFO ] root -   False Positives: 13
[INFO ] root -   False Negatives: 28
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.959
[INFO ] root -   Precision: 0.959
[INFO ] root -   Recall:    0.916
[INFO ] root -   F1-Score:  0.937
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 2467 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	2128	|	86.26%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	129	|	5.23%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	85	|	3.45%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	26	|	1.05%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	21	|	0.85%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	11	|	0.45%
[INFO ] PyDI.entitymatching.evaluat

Eval results before clustering:
  precision: 0.9591194968553459
  recall: 0.9159159159159159
  f1: 0.9370199692780338
  accuracy: 0.959
  true_positives: 305
  false_positives: 13
  false_negatives: 28
  true_negatives: 654
  threshold_used: 0.0
  total_correspondences: 4578
  filtered_correspondences: 4578
  evaluation_timestamp: 2026-05-06T18:16:03.644331
  output_files: ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/debug_results_entity_matching/matching_evaluation_summary.json', '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/debug_results_entity_matching/matching_detailed_results.csv']


[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	3110	|	100.00%
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  272
[INFO ] root -   True Negatives:  664
[INFO ] root -   False Positives: 3
[INFO ] root -   False Negatives: 61
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.936
[INFO ] root -   Precision: 0.989
[INFO ] root -   Recall:    0.817
[INFO ] root -   F1-Score:  0.895


Eval results after clustering:
  precision: 0.9890909090909091
  recall: 0.8168168168168168
  f1: 0.8947368421052633
  accuracy: 0.936
  true_positives: 272
  false_positives: 3
  false_negatives: 61
  true_negatives: 664
  threshold_used: 0.0
  total_correspondences: 3110
  filtered_correspondences: 3110
  evaluation_timestamp: 2026-05-06T18:16:03.946146
  output_files: ['/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/debug_results_entity_matching/matching_evaluation_summary.json', '/Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/debug_results_entity_matching/matching_detailed_results.csv']


## Part 4: Data Fusion

In [34]:
mbrainz["mbrainz_id"] = mbrainz["id"]

# Assign trust scores to datasets
mbrainz.attrs["trust_score"] = 3
discogs.attrs["trust_score"] = 1
lastfm.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2l], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 8,475


## Step 1: Define Fusion Strategy 

In [35]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, prefer_higher_trust, maximum

strategy = DataFusionStrategy('music_fusion_strategy')

def prefer_track_list_by_source(values, *, sources=None, source_datasets=None, **kwargs):
    """Pick one coherent track list instead of unioning near-duplicate track titles."""
    source_priority = ["musicbrainz", "discogs", "lastfm"]
    candidates = []
    for value, record_id in zip(values, sources or []):
        tracks = parse_track_list(value)
        if not tracks:
            continue
        dataset = (source_datasets or {}).get(record_id, "unknown")
        priority = source_priority.index(dataset) if dataset in source_priority else len(source_priority)
        candidates.append((priority, -len(tracks), dataset, record_id, tracks))

    if not candidates:
        return None, 0.0, {"rule": "prefer_track_list_by_source", "reason": "no_valid_track_lists"}

    priority, _, dataset, record_id, tracks = sorted(candidates)[0]
    confidence = 1.0 if priority == 0 else max(0.5, 1.0 - 0.2 * priority)
    return tracks, confidence, {
        "rule": "prefer_track_list_by_source",
        "selected_dataset": dataset,
        "selected_record_id": record_id,
        "num_tracks": len(tracks),
    }

strategy.add_attribute_fuser('name', shortest_string)
strategy.add_attribute_fuser('artist', longest_string)
strategy.add_attribute_fuser('release-date', prefer_higher_trust)
strategy.add_attribute_fuser('release-country', prefer_higher_trust)
strategy.add_attribute_fuser('duration', maximum)
strategy.add_attribute_fuser('tracks', prefer_track_list_by_source)
strategy.add_attribute_fuser('label', longest_string)


[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'shortest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'artist' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-date' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-country' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'duration' using rule 'maximum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'tracks' using rule 'prefer_track_list_by_source'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'label' using rule 'longest_string'


## Step 2: Run Fusion

In [41]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[mbrainz, discogs, lastfm],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)

def musicbrainz_cluster_id(sources):
    if not isinstance(sources, (list, tuple, set)):
        return None
    for source_id in sources:
        source_id = str(source_id)
        if source_id.startswith("mbrainz_"):
            return source_id
    return None

# Align fused records with the gold fusion sets, which use the MusicBrainz id.
# Example: [mbrainz_1, discogs_3, lastFM_1] should evaluate as id == mbrainz_1.
fused["id"] = fused["_fusion_sources"].apply(musicbrainz_cluster_id)
fused = fused.dropna(subset=["id"]).copy()

print(f'Fused rows: {len(fused):,}')
display(fused.head(5))


[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'music_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 11557 of 11557 unique IDs
[INFO ] PyDI.fusion.engine - Created 29322 record groups from 8475 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 29322 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	1849	|	6.31%
[INFO ] PyDI.fusion.engine - 		3	|	1314	|	4.48%
[INFO ] PyDI.fusion.engine - 		4	|	179	|	0.61%
[INFO ] PyDI.fusion.engine - 		5	|	90	|	0.31%
[INFO ] PyDI.fusion.engine - 		6	|	56	|	0.19%
[INFO ] PyDI.fusion.engine - 		7	|	

Fused rows: 3,624


,_id,_fusion_sources,_fusion_source_datasets,artist,duration,genre,id,label,mbrainz_id,name,name_longest_token,release-country,release-date,tracks,_fusion_confidence,_fusion_metadata
0,discogs_3,"[mbrainz_1, discogs_3, lastFM_1]","[musicbrainz, discogs, lastfm]",John B,1055.0,Electronic,mbrainz_1,New Identity Recordings,mbrainz_1,Fermats Theorem / Sight Beyond,Fermats,United Kingdom,1996-01-01,"[Fermats Theorem, Sight Beyond]",0.700000,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
1,discogs_111970,"[mbrainz_2, mbrainz_455, mbrainz_19993, mbrain...","[musicbrainz, musicbrainz, musicbrainz, musicb...",Tech Level 2,2510.0,Rock,mbrainz_2,Renegade Hardware,mbrainz_19993,Tempest,Tempest,United Kingdom,1998-12-14,"[Coma Burn, Engravings, Tempest]",0.505882,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
2,discogs_5,"[mbrainz_3, discogs_5]","[musicbrainz, discogs]",Lypid,2384.0,Electronic,mbrainz_3,Statra Recordings,mbrainz_3,The Sign's Alive,Sign's,United States,2000-09-05,"[The Sign's Alive (original mix), The Sign's A...",0.700000,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
3,discogs_101993,"[mbrainz_4, discogs_6, discogs_101993, lastFM_...","[musicbrainz, discogs, discogs, lastfm, lastfm]",Petalpusher,1632.0,Electronic,mbrainz_4,Naked Music Recordings,mbrainz_4,Surrender,Surrender,United States,1999-04-27,"[Surrender (Petalpusher original), Surrender (...",0.650000,"{'_id_rule': 'first_non_null', '_id_inputs': [..."
4,discogs_11,"[mbrainz_6, discogs_11]","[musicbrainz, discogs]","Garnier, Laurent",5938.0,Electronic,mbrainz_6,F Communications,mbrainz_6,Unreasonable Behaviour,Unreasonable,France,2000-07-24,"[The Warning, City Sphere, Forgotten Thoughts,...",0.703125,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


## Step 3: Evaluate Data Fusion

In [42]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("artist", tokenized_match)
strategy.add_evaluation_function("duration", numeric_tolerance_match, tolerance=10) # 10 seconds tolerance
strategy.add_evaluation_function("release-date", year_only_match)
strategy.add_evaluation_function("release-country", tokenized_match)
strategy.add_evaluation_function("label", tokenized_match)
strategy.add_evaluation_function("tracks", tokenized_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'artist'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'duration' with params {'tolerance': 10}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-date'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'label'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'tracks'


In [45]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')

# normalize country names in test set reusing mapping from discogs dataset
translator = SchemaTranslator()
spec.set_column("release-country", country_format="name")
mapping = discogs_mapping.copy()
mapping["source_dataset"] = "fusion_test_set"
mapping["source_column"] = mapping["target_column"] # columns already have the correct name
fusion_test_set = translator.translate(
    fusion_test_set, mapping,
    normalize=spec, on_failure="keep"
)

if "tracks" in fusion_test_set.columns:
    fusion_test_set["tracks"] = fusion_test_set["tracks"].apply(parse_track_list)

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[WARNING] root - Column 'genre' not found in dataset 'fusion_test_set'
[INFO ] root - Translating 8 columns for 'fusion_test_set'
[WARNING] PyDI.normalization.transform - Column 'genre' not found in DataFrame
[INFO ] root - Normalization complete: 69 values transformed, 0 values failed
[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/aaronsteiner/Documents/GitHub/PyDI/usecases/music/output/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.643 overall accuracy (99/154)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 55 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	tracks                           |      13 |     23.64%%
[INFO

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.643
  macro_accuracy: 0.642
  num_evaluated_records: 23
  num_evaluated_attributes: 7
  total_evaluations: 154
  total_correct: 99
  label_accuracy: 0.625
  label_count: 16
  tracks_accuracy: 0.435
  tracks_count: 23
  artist_accuracy: 0.565
  artist_count: 23
  release-date_accuracy: 0.652
  release-date_count: 23
  duration_accuracy: 0.739
  duration_count: 23
  name_accuracy: 0.739
  name_count: 23
  release-country_accuracy: 0.739
  release-country_count: 23

Overall Accuracy: 64.3%


In [46]:
fusion_test_set

,id,name_provenance,name,artist_provenance,artist,release-date_provenance,release-date,release-country_provenance,release-country,duration_provenance,duration,label_provenance,label,genre_provenance,tracks_provenance,tracks
0,mbrainz_6891,lastFM_14591,Ascension Side,mbrainz_6891,Pep Love,mbrainz_6891+discogs_30446,2003-01-01,mbrainz_6891,United States,discogs_30446,2798,discogs_30446,Hiero Imperium,,lastFM_14591+mbrainz_6891+discogs_30446,"[Chance, Sabotage, Warrior Poets, Relief, The ..."
1,mbrainz_5095,lastFM_10595,Loves Are Like Empires,lastFM_10595,Glissando,mbrainz_5095,2007-01-01,mbrainz_5095,United Kingdom,lastFM_10595,4313,,NaN,,mbrainz_5095+lastFM_10595,"[...and Before We Knew It Had Happened At All,..."
2,mbrainz_1834,mbrainz_1834,Beyond the Gates,mbrainz_1834,Cans,mbrainz_1834,2004-04-26,mbrainz_1834,Finland,mbrainz_1834,3080,discogs_8457,Noise International,,mbrainz_1834+discogs_8457,"[Fields of Yesterday, Soul Collector, Red Ligh..."
3,mbrainz_2222,discogs_9336,Guns At Dawn,discogs_9336,Baron|Pendulum (3),mbrainz_2222+discogs_9336,2005-04-18,mbrainz_2222,United Kingdom,mbrainz_2222,614,discogs_9336,Breakbeat Kaos,,mbrainz_2222+discogs_9336,"[Guns at Dawn, Ratpack]"
4,mbrainz_25432,mbrainz_25432,Up in Dreamland,mbrainz_25432,"Romani, Graziano",mbrainz_25432,2003-10-01,mbrainz_25432,Italy,mbrainz_25432,3636,,NaN,,mbrainz_25432,"[Let's Come Alive, Face the World, Every Road ..."
5,mbrainz_2807,discogs_12243,Visions And Dreams,mbrainz_2807,"Duc, Catherine",discogs_12243+mbrainz_2807,2005-01-01,mbrainz_2807,Australia,lastFM_5959,2494,discogs_12243,The Orchard,,discogs_12243+mbrainz_2807+lastFM_5959,"[Essence Of Dreams, Dancing In The Mist, Evoca..."
6,mbrainz_9448,discogs_67427,The Sign Of The Jackal,discogs_67427,Damien Thorne,mbrainz_9448,2003-06-20,mbrainz_9448,Germany,mbrainz_9448,3076,discogs_67427,Metal For Muthas,,discogs_67427+mbrainz_9448,"[The Sign Of The Jackal, Fear Of The Dark, The..."
7,mbrainz_10613,discogs_48169,Butterfly,discogs_48169,Deepfunk,discogs_48169+mbrainz_10613,2010-04-15,discogs_48169,Germany,discogs_48169,868,discogs_48169,Piemont Records,,discogs_48169+mbrainz_10613,"[Butterfly (Original Mix), Butterfly (Ryan Dav..."
8,mbrainz_15458,discogs_71795,Partys Over Los Angeles,discogs_71795,ZZT,discogs_71795+mbrainz_15458,2011-10-25,discogs_71795,Canada,discogs_71795,1303,discogs_71795,Turbo,,discogs_71795+mbrainz_15458,"[Partys Over Los Angeles, Partys Over Los Ange..."
9,mbrainz_3787,lastFM_7870,Butterfly,mbrainz_3787,"de Costa, Franklin",mbrainz_3787,2006-01-01,mbrainz_3787,Germany,mbrainz_3787,722,,NaN,,lastFM_7870+mbrainz_3787,"[Butterfly, First Wall]"
